# Standalone causal-intervention comparison

This notebook compares three distinct methods on one Architecture × Objective × Board-size condition: **Li et al. activation optimization**, **Nanda et al. linear branch patching**, and the earlier **adapted thesis hybrid**. It does not run or rewrite the common thesis evaluation. Existing common-evaluation artifacts remain unchanged.

The published experiments are 8×8 Transformer-AR. Other objectives, Mamba, and larger boards are reported explicitly as extensions.

In [ ]:
# Change only these experiment choices.
ARCHITECTURE = "transformer"  # "transformer" or "mamba"
OBJECTIVE = "ar"              # "ar" or "jepa"
BOARD_SIZE = 8                   # 8, 12, or 16
RUN_PROFILE = "smoke"          # first "smoke", then "full"
METHODS = ("li", "nanda", "adapted")

ALLOW_SOURCE_DRIFT = False
FORCE = False
DRY_RUN = False

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final")
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Master_Thesis_Artifacts")
assert (PROJECT_ROOT / "src" / "othello_thesis").is_dir(), PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT / "src"))

architecture = ARCHITECTURE.strip().lower()
objective = OBJECTIVE.strip().lower()
assert architecture in {"transformer", "mamba"}
assert objective in {"ar", "jepa"}
assert BOARD_SIZE in {8, 12, 16}
assert RUN_PROFILE in {"smoke", "full"}
assert METHODS and set(METHODS) <= {"li", "nanda", "adapted"}

if architecture == "mamba":
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "packaging", "ninja",
            "causal-conv1d==1.6.2.post1",
            "mamba-ssm==2.3.2.post1",
            "--no-build-isolation",
        ],
        check=True,
    )

from othello_thesis.evaluation.causal_suite import causal_claim_scope
print("Condition scope:", causal_claim_scope(architecture, objective, BOARD_SIZE))
print("Result claim:", "none (pipeline smoke)" if RUN_PROFILE == "smoke" else "full causal evaluation")

## Cost and resume contract

Run `smoke` first. It uses small probe/case budgets and 10 Li optimization steps only to validate the pipeline. `full` uses Li's released 1,000-step optimizer and 1,000 held-out interventions. Li is the expensive method, runs last, prints a projection after each batch, and resumes per batch after a Colab disconnect. Nanda and adapted results are saved before Li starts.

In [ ]:
def run_streaming(command):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read1(8192)
        if chunk:
            print(chunk.decode("utf-8", errors="replace"), end="", flush=True)
        elif process.poll() is not None:
            break
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

command = [
    sys.executable, "-u", str(PROJECT_ROOT / "scripts" / "run_causal_intervention.py"),
    "--architecture", architecture,
    "--objective", objective,
    "--board-size", str(BOARD_SIZE),
    "--profile", RUN_PROFILE,
    "--methods", *METHODS,
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--project-root", str(PROJECT_ROOT),
]
if ALLOW_SOURCE_DRIFT:
    command.append("--allow-source-drift")
if FORCE:
    command.append("--force")
if DRY_RUN:
    command.append("--dry-run")
run_streaming(command)

## Outputs

The suite writes only below the selected run's `causal_intervention/causal_intervention_suite_v1/{smoke|full}/` directory. Each method is resumable, the Li progress directory is batch-granular, and the final Markdown summary labels replications versus extensions.